# ResNet-18 image classification

Choose a bundled picture or upload your own. The host exported TorchVision
ResNet-18 through PyTorch (BatchNorm folded, our torchao W8A8 config,
`torch.export`, torch-mlir) and compiled it with `npu-compile`; this notebook
loads the package with `src/runtime`. Each code block reports its wall time;
time spent choosing a picture between blocks is excluded.


## 1. Find the deployed bundle


In [ ]:
import json
import os
from pathlib import Path
import sys
import time
import numpy as np

block_seconds = {}
_started = time.perf_counter()
start = Path.cwd().resolve()
release_root = next((p for p in (start, *start.parents) if (p / 'src/runtime').is_dir()), None)
if release_root is None:
    raise RuntimeError('Open the notebook from the deployed bundle or the repository')
sys.path.insert(0, str(release_root))
in_bundle = (release_root / 'package/manifest.json').is_file()
model_dir = release_root / ('model' if in_bundle else 'dist/resnet18/model')
package_dir = Path(os.environ.get('NPU_RESNET18_PACKAGE',
                                  str(release_root / ('package' if in_bundle else 'dist/resnet18/package'))))
backend = os.environ.get('NPU_RESNET18_BACKEND', 'pynq')
print(f'package: {package_dir}; backend: {backend}')
block_seconds['1_connect'] = time.perf_counter() - _started
print(block_seconds)


## 2. Program the FPGA and load the compiled package

The overlay must implement the ISA front end (ABI 1.1). The first load builds
libnpu_rt with the board's compiler. Host simulation can be selected with
`NPU_RESNET18_BACKEND=sim`.


In [ ]:
_started = time.perf_counter()
from src.runtime.tasks.classifier import ImageEngine, preprocess, top_predictions
os.environ['XILINX_XRT'] = '/usr'
overlay = None
if backend == 'pynq':
    from pynq import Overlay
    bitstream = Path(os.environ.get('NPU_RESNET18_OVERLAY', str(release_root / 'npu_matrix.bit')))
    if not bitstream.is_file() or not bitstream.with_suffix('.hwh').is_file():
        raise FileNotFoundError(f'Matching BIT/HWH pair is missing: {bitstream}')
    print(f'overlay: {bitstream}')
    overlay = Overlay(str(bitstream))
engine = ImageEngine(package_dir, backend, overlay=overlay)
block_seconds['2_overlay_and_package'] = time.perf_counter() - _started
print(block_seconds)


## 3. Choose a picture

Pick one of the bundled pictures from the dropdown, or upload your own. An
upload always wins over the dropdown. The bundled set is five Creative Commons
photographs fetched by `python -m src.deploy.resnet18_assets gallery`; each one declares the
ImageNet class it should produce, so Step 6 can say whether the network got it
right.

In [ ]:
_started = time.perf_counter()
import ipywidgets as widgets
from IPython.display import display

gallery_path = model_dir / 'gallery-source.json'
if not gallery_path.is_file():
    gallery_path = release_root / 'src/deploy/resnet18-gallery-source.json'
gallery = []
if gallery_path.is_file():
    for entry in json.loads(gallery_path.read_text(encoding='utf-8'))['images']:
        candidate = model_dir / entry['filename']
        if candidate.is_file():
            gallery.append((entry['expected_class'], candidate, entry))

options = [
    (f"{expected}   ({path.name})", index)
    for index, (expected, path, _entry) in enumerate(gallery)
]
options.append(('-- upload my own picture --', -1))

chooser = widgets.Dropdown(
    options=options,
    value=0 if gallery else -1,
    description='picture:',
    layout=widgets.Layout(width='460px'),
)
uploader = widgets.FileUpload(accept='image/*', multiple=False)
display(chooser, uploader)

print(f'{len(gallery)} bundled pictures available.')
if not gallery:
    print('Run python -m src.deploy.resnet18_assets gallery to fetch them, or upload your own.')
print('An upload overrides the dropdown. Then run the next cell.')
block_seconds['3_choose_picture'] = time.perf_counter() - _started
print(block_seconds)

## 4. Preprocess on the board and preview the input

The left panel is the original image. The right panel is the normalized NHWC
float32 package input converted back to RGB for inspection. Activation INT8
quantization is part of the compiled model.

In [ ]:
_started = time.perf_counter()
from io import BytesIO
from PIL import Image
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
def uploaded_file(widget):
    """Return (name, bytes) for ipywidgets 7 or 8, or None when nothing was picked."""
    value = widget.value
    if not value:
        return None
    if isinstance(value, dict):
        name, payload = next(iter(value.items()))
        return name, bytes(payload['content'])
    entry = value[0]
    return entry['name'], bytes(entry['content'])


picked = uploaded_file(uploader)
if picked is not None:
    name, content = picked
    image_path = BytesIO(content)
    expected_class = None
    image_label = f'{name}   (uploaded, {len(content):,} bytes)'
elif chooser.value >= 0:
    expected_class, image_path, entry = gallery[chooser.value]
    image_label = (
        f"{image_path.name}   {entry['license']['spdx']}, "
        f"{entry['license']['attribution']}"
    )
else:
    image_path = model_dir / 'demo-image.jpg'
    expected_class = 'Samoyed'
    image_label = f'{image_path.name}   (bundled sample)'


with Image.open(image_path) as image:
    original = image.convert('RGB')
    npu_input = preprocess(original, engine.contract)
preview = np.clip(npu_input * np.asarray(engine.contract['std']) +
                  np.asarray(engine.contract['mean']), 0, 1)
figure, axes = plt.subplots(1, 2, figsize=(9, 5))
axes[0].imshow(original)
axes[0].set_title('the picture')
axes[1].imshow(preview)
axes[1].set_title('compiled package input: 224 x 224 float32')
for axis in axes:
    axis.set_xticks([])
    axis.set_yticks([])
figure.suptitle(image_label)
figure.tight_layout()
plt.show()
print(f'input tensor: {npu_input.shape} {npu_input.dtype}')
block_seconds['4_preprocess_and_preview'] = time.perf_counter() - _started
print(block_seconds)

## 5. Run the compiled model

With the `pynq` backend, a positive job count confirms work reached the FPGA.
The `sim` backend reports simulated work and is host evidence only.

In [ ]:
_started = time.perf_counter()
logits = engine.predict(npu_input)
block_seconds['5_inference'] = time.perf_counter() - _started
if backend == 'pynq':
    assert engine.last_stats['jobs'] > 0, 'No physical NPU job executed'
print(f'backend: {backend}; NPU counters: {engine.last_stats}')
print(block_seconds)

## 6. Read the prediction

The top-5 ImageNet classes, highest first. For a bundled picture the expected
class is known, so the notebook states `CORRECT` or `INCORRECT`. For your own
upload there is no ground truth: look at the picture in Step 4 and judge it.

In [ ]:
_started = time.perf_counter()
class_names = (model_dir / 'imagenet-classes.txt').read_text(encoding='utf-8').splitlines()
top5 = top_predictions(logits, class_names)
for rank, prediction in enumerate(top5, start=1):
    print(f'{rank}. {prediction.name:<34} {prediction.probability:7.2%}')
print()
print(f'predicted: {top5[0].name}   (index {top5[0].index})')
if expected_class is None:
    print('expected:  not declared for an uploaded picture; judge it yourself')
else:
    verdict = 'CORRECT' if top5[0].name == expected_class else 'INCORRECT'
    print(f'expected:  {expected_class}')
    print(f'verdict:   {verdict}')

labels = [prediction.name for prediction in reversed(top5)]
values = [prediction.probability for prediction in reversed(top5)]
colors = ['#86b6ef'] * (len(values) - 1) + ['#2a78d6']

figure, axis = plt.subplots(figsize=(8, 3.2), facecolor='#fcfcfb')
axis.set_facecolor('#fcfcfb')
bars = axis.barh(labels, values, color=colors, height=0.62)
for bar, value in zip(bars, values):
    axis.text(
        value + max(values) * 0.02,
        bar.get_y() + bar.get_height() / 2,
        f'{value:.1%}',
        va='center',
        color='#52514e',
        fontsize=10,
    )
axis.set_xlim(0, max(values) * 1.22)
axis.set_title('top-5 ImageNet prediction', color='#0b0b0b', loc='left', pad=12)
axis.tick_params(colors='#52514e', length=0)
axis.xaxis.set_visible(False)
for side in ('top', 'right', 'bottom'):
    axis.spines[side].set_visible(False)
axis.spines['left'].set_color('#d8d7d2')
figure.tight_layout()
plt.show()
block_seconds['6_prediction'] = time.perf_counter() - _started
print(json.dumps({'block_seconds': block_seconds,
                  'total_block_seconds': sum(block_seconds.values()),
                  'backend': backend, 'npu': engine.last_stats,
                  'top5': [vars(prediction) for prediction in top5],
                  'expected_class': expected_class,
                  'correct': top5[0].name == expected_class if expected_class else None}, indent=2))
# Re-running a block replaces its duration; this total excludes human wait time.